In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load preprocessed FCT data and RQ1/RQ2 results
df             = pd.read_csv("fct_cleaned.csv", low_memory=False)
outcome_matrix = pd.read_csv("fct_outcome_matrix.csv", index_col=0)
cost_vector    = pd.read_csv("fct_cost_vector.csv", index_col=0, header=0)
cost_vector.columns = ['mean_cost_s']
unit_summary   = pd.read_csv("fct_unit_summary.csv")
cred_df        = pd.read_csv("fct_rq2_cred.csv")

print("All files loaded successfully")
print(f"Cred steps: {len(cred_df)}")

In [ ]:
outcome_matrix = pd.read_csv("fct_outcome_matrix.csv", index_col=0)
cost_vector    = pd.read_csv("fct_cost_vector.csv", index_col=0, header=0)
cost_vector.columns = ['mean_cost_s']
unit_summary   = pd.read_csv("fct_unit_summary.csv")
cred_df        = pd.read_csv("fct_rq2_cred.csv")
df             = pd.read_csv("fct_cleaned.csv", low_memory=False)

if 'unit_id' not in df.columns:
    df['unit_id'] = df['source_file'].astype(str).str.strip()

C_red       = cred_df['test_id'].tolist()
C_full      = outcome_matrix.columns.tolist()
C_full_cost = cost_vector['mean_cost_s'].sum()
C_red_cost  = sum(
    cost_vector.loc[s, 'mean_cost_s']
    for s in C_red
    if s in cost_vector.index
)

# MAB hyperparameters
delta                 = 85 / 1_000_000
ESCAPE_PENALTY        = 10
PASS_RATE_WINDOW      = 50
PASS_RATE_THRESHOLD   = 0.93
MIN_UNSTABLE_DURATION = 100

# Separate genuine defects from equipment aborts
all_failing = unit_summary[
    unit_summary['overall_outcome'] == 0
]['unit_id'].tolist()

genuine_fail_units = []
abort_only_units   = []
for unit_id in all_failing:
    unit_steps    = df[df['unit_id'] == unit_id]
    has_step_fail = (unit_steps['result'] == 'FAIL').any()
    if has_step_fail:
        genuine_fail_units.append(unit_id)
    else:
        abort_only_units.append(unit_id)

failing_units = genuine_fail_units
N_fail        = len(failing_units)

print(f"Total units:          {len(unit_summary)}")
print(f"Genuine FAIL units:   {N_fail}")
print(f"ABORT-only excluded:  {len(abort_only_units)}")
print(f"Full sequence cost:   {C_full_cost:.2f} s")
print(f"Cred cost:            {C_red_cost:.2f} s")
print(f"Cred size:            {len(C_red)} / {len(C_full)} steps")

In [ ]:
# Build chronological unit stream — preserving temporal order
seen        = set()
unit_stream = []
for u in df['unit_id'].tolist():
    if u not in seen:
        unit_stream.append(u)
        seen.add(u)

print(f"Units in stream: {len(unit_stream)}")

In [ ]:
# Compute rolling pass rate stability signal
unit_outcome_map = unit_summary.set_index(
    'unit_id'
)['overall_outcome'].to_dict()

unit_pass_seq = np.array([
    unit_outcome_map.get(u, 1)
    for u in unit_stream
])

rolling_pass_rate = np.zeros(len(unit_stream))
for i in range(len(unit_stream)):
    window_start = max(0, i - PASS_RATE_WINDOW + 1)
    window_vals  = unit_pass_seq[window_start:i+1]
    rolling_pass_rate[i] = window_vals.mean()

print(f"Rolling pass rate stats:")
print(f"  Mean:       {rolling_pass_rate.mean():.4f}")
print(f"  Min:        {rolling_pass_rate.min():.4f}")
print(f"  Max:        {rolling_pass_rate.max():.4f}")
print(f"  % stable:   "
      f"{(rolling_pass_rate >= PASS_RATE_THRESHOLD).mean()*100:.1f}%")
print(f"  % unstable: "
      f"{(rolling_pass_rate < PASS_RATE_THRESHOLD).mean()*100:.1f}%")

In [ ]:
# MAB reward function — aligns agent behaviour 
# with quality-cost objective
def unit_is_defective(unit_id):
    return unit_id in failing_units

def cred_detects(unit_id):
    if unit_id not in outcome_matrix.index:
        return True
    row = outcome_matrix.loc[unit_id]
    for step in C_red:
        if step in row.index and row[step] == 0:
            return True
    return False

def get_reward(chosen_arm, unit_id):
    is_defective = unit_is_defective(unit_id)
    detected     = False
    escaped      = False

    if chosen_arm == 0:
        # Full plan — safe but costly
        reward = 0.0
        if is_defective:
            detected = True
    else:
        if not is_defective:
            # Clean unit under reduced plan
            reward = 1.0
        elif cred_detects(unit_id):
            # Defect detected by reduced plan
            reward   = 0.5
            detected = True
        else:
            # Defect escaped reduced plan
            reward  = -ESCAPE_PENALTY
            escaped = True

    return reward, detected, escaped

print("Helper functions defined")

In [ ]:
# Thompson Sampling MAB training loop
# Beta posterior updated after each unit
alpha_ts = [5.0, 1.0]  # conservative prior for Cfull
beta_ts  = [1.0, 1.0]  # uniform prior for Cred

choices_ts          = []
escape_ts           = 0
defects_ts          = 0
cost_ts             = []
stability_log       = []
rewards_log         = []
escape_log          = []
stability_overrides = 0

np.random.seed(42)

for i, unit_id in enumerate(unit_stream):
    pass_rate = rolling_pass_rate[i]
    stable    = pass_rate >= PASS_RATE_THRESHOLD
    stability_log.append(1 if stable else 0)

    theta_full = np.random.beta(alpha_ts[0], beta_ts[0])
    theta_red  = np.random.beta(alpha_ts[1], beta_ts[1])

    if not stable:
        theta_full += (PASS_RATE_THRESHOLD - pass_rate) * 10
        stability_overrides += 1

    chosen = 1 if theta_red > theta_full else 0
    choices_ts.append(chosen)

    reward, detected, escaped = get_reward(chosen, unit_id)

    cost_ts.append(C_full_cost if chosen == 0 else C_red_cost)

    if detected:
        defects_ts += 1
    if escaped:
        escape_ts += 1

    reward_clipped = max(0.0, min(1.0, reward))
    if reward_clipped >= 0.5:
        alpha_ts[chosen] += 1
    else:
        beta_ts[chosen]  += 1

    rewards_log.append(reward_clipped)
    escape_log.append(escape_ts)

print("Thompson Sampling complete")
print(f"Chose Cred:  {choices_ts.count(1)} "
      f"({choices_ts.count(1)/len(choices_ts)*100:.1f}%)")
print(f"Escaped:     {escape_ts}")

In [ ]:
# Upper Confidence Bound (UCB) training loop
counts_ucb      = [1, 1]
rewards_ucb_cum = [0.0, 0.0]

choices_ucb = []
escape_ucb  = 0
defects_ucb = 0
cost_ucb    = []

np.random.seed(42)

for i, unit_id in enumerate(unit_stream):
    pass_rate = rolling_pass_rate[i]
    stable    = pass_rate >= PASS_RATE_THRESHOLD
    t         = i + 1

    ucb_full = (rewards_ucb_cum[0] / counts_ucb[0] +
                np.sqrt(2 * np.log(t) / counts_ucb[0]))
    ucb_red  = (rewards_ucb_cum[1] / counts_ucb[1] +
                np.sqrt(2 * np.log(t) / counts_ucb[1]))

    if not stable:
        ucb_full += (PASS_RATE_THRESHOLD - pass_rate) * 10

    chosen = 1 if ucb_red > ucb_full else 0
    choices_ucb.append(chosen)

    reward, detected, escaped = get_reward(chosen, unit_id)

    cost_ucb.append(C_full_cost if chosen == 0 else C_red_cost)

    if detected:
        defects_ucb += 1
    if escaped:
        escape_ucb += 1

    reward_clipped          = max(0.0, min(1.0, reward))
    rewards_ucb_cum[chosen] += reward_clipped
    counts_ucb[chosen]      += 1

print("UCB complete")
print(f"Chose Cred:  {choices_ucb.count(1)} "
      f"({choices_ucb.count(1)/len(choices_ucb)*100:.1f}%)")
print(f"Escaped:     {escape_ucb}")

In [ ]:
# Epsilon-Greedy training loop
EPSILON         = 0.1
mean_rewards_eg = [0.0, 0.0]
counts_eg       = [1, 1]

choices_eg = []
escape_eg  = 0
defects_eg = 0
cost_eg    = []

np.random.seed(42)

for i, unit_id in enumerate(unit_stream):
    pass_rate = rolling_pass_rate[i]
    stable    = pass_rate >= PASS_RATE_THRESHOLD

    if np.random.random() < EPSILON:
        chosen = np.random.randint(0, 2)
    else:
        scores = mean_rewards_eg.copy()
        if not stable:
            scores[0] += (PASS_RATE_THRESHOLD - pass_rate) * 10
        chosen = int(np.argmax(scores))

    choices_eg.append(chosen)

    reward, detected, escaped = get_reward(chosen, unit_id)

    cost_eg.append(C_full_cost if chosen == 0 else C_red_cost)

    if detected:
        defects_eg += 1
    if escaped:
        escape_eg += 1

    reward_clipped      = max(0.0, min(1.0, reward))
    mean_rewards_eg[chosen] = (
        mean_rewards_eg[chosen] * counts_eg[chosen] +
        reward_clipped
    ) / (counts_eg[chosen] + 1)
    counts_eg[chosen] += 1

print("Epsilon-Greedy complete")
print(f"Chose Cred:  {choices_eg.count(1)} "
      f"({choices_eg.count(1)/len(choices_eg)*100:.1f}%)")
print(f"Escaped:     {escape_eg}")

In [ ]:
# Compute and compare results across all algorithms
def compute_metrics(choices, cost_log,
                    escaped, defects, label):
    total    = len(choices)
    pct_cred = choices.count(1) / total * 100
    cost_mu  = np.mean(cost_log)
    saving   = (C_full_cost - cost_mu) / C_full_cost * 100
    esc_risk = escaped / N_fail if N_fail > 0 else 0
    return {
        'Algorithm':     label,
        'Cred (%)':      round(pct_cred, 1),
        'Cost/unit (s)': round(cost_mu, 2),
        'Saving (%)':    round(saving, 2),
        'Escaped':       escaped,
        'Escape Risk':   round(esc_risk, 6)
    }

rows = [
    {
        'Algorithm':     'Baseline (Cfull)',
        'Cred (%)':      0.0,
        'Cost/unit (s)': round(C_full_cost, 2),
        'Saving (%)':    0.0,
        'Escaped':       0,
        'Escape Risk':   0.0
    },
    {
        'Algorithm':     'Static Cred (RQ1)',
        'Cred (%)':      100.0,
        'Cost/unit (s)': round(C_red_cost, 2),
        'Saving (%)':    round(
            (C_full_cost - C_red_cost) / C_full_cost * 100, 2
        ),
        'Escaped':       0,
        'Escape Risk':   0.0
    },
    compute_metrics(
        choices_eg, cost_eg,
        escape_eg, defects_eg, 'Epsilon-Greedy'
    ),
    compute_metrics(
        choices_ucb, cost_ucb,
        escape_ucb, defects_ucb, 'UCB'
    ),
    compute_metrics(
        choices_ts, cost_ts,
        escape_ts, defects_ts, 'Thompson Sampling'
    ),
]

results_df = pd.DataFrame(rows)
results_df.to_csv("fct_results_comparison.csv", index=False)

print(f"\n{'='*75}")
print("RESULTS TABLE — FCT DATASET")
print(f"{'='*75}")
print(results_df.to_string(index=False))

In [ ]:
rewards_arr   = np.array(rewards_log)
choices_arr   = np.array(choices_ts)
costs_arr     = np.array(cost_ts)
stability_arr = np.array(stability_log)
window        = 500
alpha_ema     = 0.0005
exec_index    = np.arange(len(rewards_arr))

rolling_mean = pd.Series(rewards_arr).rolling(
    window=window, min_periods=1
).mean().values

ema    = np.zeros(len(rewards_arr))
ema[0] = rolling_mean[min(window, len(rolling_mean)-1)]
for i in range(1, len(rewards_arr)):
    ema[i] = alpha_ema * rewards_arr[i] + (1 - alpha_ema) * ema[i-1]

rolling_cred_ts = pd.Series(choices_arr).rolling(
    window=window, min_periods=1
).mean().values

rolling_cred_ucb = pd.Series(
    np.array(choices_ucb)
).rolling(window=window, min_periods=1).mean().values

rolling_cred_eg = pd.Series(
    np.array(choices_eg)
).rolling(window=window, min_periods=1).mean().values

cumulative_avg_cost = np.cumsum(costs_arr) / (exec_index + 1)
cumulative_saving   = (
    C_full_cost - cumulative_avg_cost
) / C_full_cost * 100

unstable_mask = stability_arr == 0

def clean_mask(mask, min_duration):
    cleaned   = mask.copy().astype(bool)
    in_block  = False
    start_idx = 0
    for i in range(len(mask)):
        if mask[i] and not in_block:
            in_block  = True
            start_idx = i
        elif not mask[i] and in_block:
            in_block = False
            if i - start_idx < min_duration:
                cleaned[start_idx:i] = False
    if in_block and len(mask) - start_idx < min_duration:
        cleaned[start_idx:] = False
    return cleaned

unstable_mask_clean = clean_mask(
    unstable_mask, MIN_UNSTABLE_DURATION
)

pct_cred_ts     = choices_ts.count(1) / len(choices_ts) * 100
cost_per_unit   = np.mean(cost_ts)
cost_saving_pct = (
    C_full_cost - cost_per_unit
) / C_full_cost * 100

def style_ax(ax):
    ax.xaxis.set_major_locator(plt.MaxNLocator(10))
    ax.yaxis.set_major_locator(plt.MaxNLocator(10))
    ax.tick_params(axis='both', labelsize=11)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, alpha=0.3)

def shade_unstable_regions(ax, mask, exec_index, label=True):
    in_block  = False
    start_idx = 0
    blocks    = []
    for i in range(len(mask)):
        if mask[i] and not in_block:
            in_block  = True
            start_idx = i
        elif not mask[i] and in_block:
            in_block = False
            blocks.append((start_idx, i))
    if in_block:
        blocks.append((start_idx, len(mask)))
    for k, (s, e) in enumerate(blocks):
        ax.axvspan(
            exec_index[s],
            exec_index[min(e, len(exec_index)-1)],
            alpha=0.15, color='red',
            label='Unstable region' if (k==0 and label) else ''
        )
    return len(blocks)

print(f"Rolling stats computed")
print(f"Original unstable: {unstable_mask.sum()}")
print(f"Cleaned unstable:  {unstable_mask_clean.sum()}")
print(f"Cost saving (TS):  {cost_saving_pct:.2f}%")

In [ ]:
df_steps = df[
    df['result'].isin(['PASS', 'FAIL', 'ABORT'])
].copy()
df_steps['step_pass'] = (
    df_steps['result'] == 'PASS'
).astype(int)
df_steps['step_fail'] = (
    df_steps['result'].isin(['FAIL', 'ABORT'])
).astype(int)
step_idx      = np.arange(len(df_steps))
cumpass_steps = (
    df_steps['step_pass'].cumsum().values / (step_idx + 1)
)
cumfail_steps = (
    df_steps['step_fail'].cumsum().values / (step_idx + 1)
)

fig1, ax1 = plt.subplots(figsize=(7, 5))
ax1.plot(
    step_idx, cumpass_steps,
    color='black', linewidth=1.5,
    label='Cumulative pass rate (FCT)'
)
ax1.set_xlabel(
    'Execution index (FCT events in time order)',
    fontsize=14, fontweight='bold'
)
ax1.set_ylabel(
    'Cumulative pass rate',
    fontsize=14, fontweight='bold'
)
ax1.set_ylim(0.985, 1.0)
ax1.legend(fontsize=10)
style_ax(ax1)
plt.tight_layout()
plt.savefig('fct_fig1_cumpass.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fct_fig1_cumpass.png")

In [ ]:
fig2, ax2 = plt.subplots(figsize=(7, 5))
ax2.plot(
    step_idx, cumfail_steps,
    color='black', linewidth=1.5,
    label='Cumulative defect rate (FCT)'
)
ax2.set_xlabel(
    'Execution index (FCT events in time order)',
    fontsize=14, fontweight='bold'
)
ax2.set_ylabel(
    'Cumulative defect rate',
    fontsize=14, fontweight='bold'
)
ax2.legend(fontsize=10)
style_ax(ax2)
plt.tight_layout()
plt.savefig('fct_fig2_cumdefect.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fct_fig2_cumdefect.png")

In [ ]:
fig3, ax3 = plt.subplots(figsize=(7, 5))
shade_unstable_regions(ax3, unstable_mask_clean, exec_index)
ax3.plot(
    exec_index, rolling_mean,
    color='black', linewidth=2,
    label=f'Rolling mean reward (w={window})'
)
ax3.set_xlabel(
    'Execution index (FCT events in time order)',
    fontsize=14, fontweight='bold'
)
ax3.set_ylabel(
    'Average reward',
    fontsize=14, fontweight='bold'
)
ax3.set_ylim(0.90, 1.01)
ax3.legend(fontsize=10)
style_ax(ax3)
plt.tight_layout()
plt.savefig('fct_fig3_reward_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fct_fig3_reward_trend.png")

In [ ]:
fig4, ax4 = plt.subplots(figsize=(7, 5))
shade_unstable_regions(ax4, unstable_mask_clean, exec_index)
ax4.plot(
    exec_index, rolling_cred_ts,
    color='black', linewidth=2,
    label=f'Thompson Sampling (w={window})'
)
ax4.axhline(
    y=0.5, color='red', linestyle='--',
    linewidth=1.5, label='50% threshold'
)
ax4.set_xlabel(
    'Execution index (FCT events in time order)',
    fontsize=14, fontweight='bold'
)
ax4.set_ylabel(
    'Fraction selecting Cred',
    fontsize=14, fontweight='bold'
)
ax4.set_ylim(0, 1)
ax4.set_xlim(0, 5000)
ax4.legend(fontsize=10)
style_ax(ax4)
plt.tight_layout()
plt.savefig('fct_fig4_arm_selection.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fct_fig4_arm_selection.png")

In [ ]:
fig5, ax5 = plt.subplots(figsize=(7, 5))
shade_unstable_regions(ax5, unstable_mask_clean, exec_index)
ax5.plot(
    exec_index, rolling_cred_ts,
    color='black', linewidth=2,
    label='Thompson Sampling'
)
ax5.plot(
    exec_index, rolling_cred_ucb,
    color='steelblue', linewidth=1.5,
    linestyle='--', label='UCB'
)
ax5.plot(
    exec_index, rolling_cred_eg,
    color='green', linewidth=1.5,
    linestyle=':', label='Epsilon-Greedy'
)
ax5.axhline(
    y=0.5, color='red', linestyle='--',
    linewidth=1, alpha=0.5, label='50% threshold'
)
ax5.set_xlabel(
    'Execution index (FCT events in time order)',
    fontsize=14, fontweight='bold'
)
ax5.set_ylabel(
    'Fraction selecting Cred',
    fontsize=14, fontweight='bold'
)
ax5.set_ylim(-0.05, 1.05)
ax5.legend(fontsize=10)
style_ax(ax5)
plt.tight_layout()
plt.savefig('fct_fig5_algo_comparison.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fct_fig5_algo_comparison.png")

In [ ]:
fig6, ax6 = plt.subplots(figsize=(7, 5))
ax6.plot(
    exec_index, rolling_pass_rate,
    color='black', linewidth=1.5,
    label=f'Rolling pass rate (w={PASS_RATE_WINDOW})'
)
ax6.axhline(
    y=PASS_RATE_THRESHOLD, color='red',
    linestyle='--', linewidth=1.5,
    label=f'Stability threshold ({PASS_RATE_THRESHOLD})'
)

stable_mask_clean = ~unstable_mask_clean
in_block    = False
start_idx   = 0
first_label = True
for i in range(len(stable_mask_clean)):
    if stable_mask_clean[i] and not in_block:
        in_block  = True
        start_idx = i
    elif not stable_mask_clean[i] and in_block:
        in_block = False
        if i - start_idx >= 10:
            ax6.axvspan(
                exec_index[start_idx], exec_index[i-1],
                alpha=0.1, color='green',
                label='Stable' if first_label else ''
            )
            first_label = False
if in_block:
    ax6.axvspan(
        exec_index[start_idx], exec_index[-1],
        alpha=0.1, color='green',
        label='Stable' if first_label else ''
    )

shade_unstable_regions(ax6, unstable_mask_clean, exec_index)

ax6.set_xlabel(
    'Execution index (FCT events in time order)',
    fontsize=14, fontweight='bold'
)
ax6.set_ylabel(
    f'Rolling pass rate (w={PASS_RATE_WINDOW})',
    fontsize=14, fontweight='bold'
)
ax6.legend(fontsize=10)
style_ax(ax6)
plt.tight_layout()
plt.savefig('fct_fig6_rolling_passrate.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fct_fig6_rolling_passrate.png")

In [ ]:
fig7, ax7 = plt.subplots(figsize=(7, 5))
ax7.plot(
    exec_index, cumulative_saving,
    color='black', linewidth=1.5,
    label='Cumulative cost saving (%)'
)
ax7.axhline(
    y=cost_saving_pct, color='red',
    linestyle='--', linewidth=1.5,
    label=f'Converged: {cost_saving_pct:.1f}%'
)
ax7.set_xlabel(
    'Execution index (FCT events in time order)',
    fontsize=14, fontweight='bold'
)
ax7.set_ylabel(
    'Cumulative cost saving (%)',
    fontsize=14, fontweight='bold'
)
ax7.set_ylim(-1, 25)
ax7.legend(fontsize=10)
style_ax(ax7)
plt.tight_layout()
plt.savefig('fct_fig7_cost_saving.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fct_fig7_cost_saving.png")

In [ ]:
cpk_sorted = cpk_overall.dropna(
    subset=['cpk']
).sort_values('cpk', ascending=True).reset_index(drop=True)

colors_cpk = [
    'red'    if c < 1.0  else
    'orange' if c < 1.33 else
    'green'
    for c in cpk_sorted['cpk']
]

fig9, ax9 = plt.subplots(figsize=(10, 6))
ax9.barh(
    range(len(cpk_sorted)),
    cpk_sorted['cpk'],
    color=colors_cpk, alpha=0.7
)
ax9.axvline(
    x=1.33, color='black', linestyle='--',
    linewidth=2, label='Capability threshold (Cpk=1.33)'
)
ax9.axvline(
    x=1.0, color='red', linestyle=':',
    linewidth=1.5, label='Minimum acceptable (Cpk=1.0)'
)
ax9.set_yticks(range(len(cpk_sorted)))
ax9.set_yticklabels(
    [t[:45] for t in cpk_sorted['test_id']], fontsize=7
)
ax9.set_xlabel(
    'Cpk value', fontsize=14, fontweight='bold'
)
ax9.set_ylabel(
    'Test step', fontsize=12, fontweight='bold'
)
ax9.set_xlim(0, 20)
ax9.legend(fontsize=10)
ax9.spines['top'].set_visible(False)
ax9.spines['right'].set_visible(False)
ax9.grid(True, alpha=0.3, axis='x')

n_capable  = (cpk_sorted['cpk'] >= 1.33).sum()
n_marginal = (
    (cpk_sorted['cpk'] >= 1.0) &
    (cpk_sorted['cpk'] < 1.33)
).sum()
n_incap    = (cpk_sorted['cpk'] < 1.0).sum()

ax9.text(
    0.98, 0.02,
    f'Capable (≥1.33): {n_capable}\n'
    f'Marginal (1.0–1.33): {n_marginal}\n'
    f'Incapable (<1.0): {n_incap}',
    transform=ax9.transAxes, fontsize=10,
    verticalalignment='bottom',
    horizontalalignment='right',
    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5)
)
plt.tight_layout()
plt.savefig('fct_fig9_cpk.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fct_fig9_cpk.png")

In [ ]:
mab_log_df = pd.DataFrame({
    'unit_id':           unit_stream,
    'chosen_arm_ts':     choices_ts,
    'chosen_arm_ucb':    choices_ucb,
    'chosen_arm_eg':     choices_eg,
    'reward_ts':         rewards_log,
    'cost_ts':           cost_ts,
    'cost_ucb':          cost_ucb,
    'cost_eg':           cost_eg,
    'escaped_so_far_ts': escape_log,
    'process_stable':    stability_log,
    'rolling_pass_rate': rolling_pass_rate.tolist(),
})
mab_log_df.to_csv("fct_mab_log.csv", index=False)
results_df.to_csv("fct_results_comparison.csv", index=False)

print("All outputs saved")
print(f"\nFinal results table:")
print(results_df.to_string(index=False))


In [ ]:

print(f"C_full cost:               {C_full_cost:.2f} s")
print(f"C_red cost:                {C_red_cost:.2f} s")
print(f"Static C_red saving:       {C_full_cost - C_red_cost:.2f} s")


ts_mean_cost = np.mean(cost_ts)
ts_saving_s  = C_full_cost - ts_mean_cost
print(f"TS (β=10) mean cost/board: {ts_mean_cost:.2f} s")
print(f"TS (β=10) saving/board:    {ts_saving_s:.2f} s")

eg_mean_cost = np.mean(cost_eg)
print(f"EG saving/board:           {C_full_cost - eg_mean_cost:.2f} s")

ucb_mean_cost = np.mean(cost_ucb)
print(f"UCB saving/board:          {C_full_cost - ucb_mean_cost:.2f} s")